# 21 · Query & federation — DuckDB two ways (embedded + GizmoSQL)

**DuckDB is an embedded, columnar OLAP engine** — SQLite's ergonomics with a
vectorized analytical core. It reads Parquet/Arrow directly, pushes filters and
projections down into the scan, and shares memory with Arrow at zero copy. In the
mesh it shows up in **two different shapes**, and this notebook runs both:

1. **Embedded** — DuckDB running *inside this kernel*, no server. It queries lakeFS
   Parquet live over `httpfs`, and does zero-copy Arrow interop with polars/pyarrow
   **in the same process**. This is the *single-process / edge / ad-hoc analytics*
   story: no network hop, no service to keep up.
2. **Served (GizmoSQL)** — the **same engine** fronted by an **Arrow Flight SQL**
   server at a real `host:port`, so many clients (BI tools, IDEs, pipelines) share
   *one* DuckDB over the network. It serves the silver as **persisted DuckDB base
   tables** and does real relational JOINs. This is the *shared / multi-client* story.

### Where this sits vs notebook `20`

The query layer has two halves. **Trino** (notebook `20`) is the **distributed-federation**
half — one SQL statement spanning *many* backing systems, cross-catalog joins, no ETL.
**DuckDB / GizmoSQL** (this notebook) is the **single-node OLAP** half — one fast
columnar engine over Parquet/base-tables, embedded in your process or served to a few
clients. Different jobs: Trino combines *many* engines; DuckDB *is* the engine.

> **Read-only.** Everything here is `SELECT` / `EXPLAIN` / `DESCRIBE`. The embedded half
> reads lakeFS `main` (never writes an object); the served half queries GizmoSQL's base
> tables only. GizmoSQL also runs each client statement in an **isolated session**, so
> client-side DDL wouldn't persist anyway — but we never attempt a write. Because we
> create nothing, there is **no cleanup section** (unlike notebooks `10` / `11`).

# Half 1 — Embedded DuckDB over lakeFS

DuckDB, polars, pyarrow and s3fs are **all baked into the singleuser image**, so this
half needs **no install** — the engine is already in the kernel. We point DuckDB's
`httpfs` extension at the lakeFS S3 gateway and read the versioned Parquet directly.

## Connect — DuckDB `httpfs` against the lakeFS S3 gateway

Connection is **env-driven**. The committed default is the **in-cluster** lakeFS service
URL (`lakefs.data-mesh.svc.cluster.local:8000`); a validation run overrides `LAKEFS_ENDPOINT`
(e.g. to a NodePort or `kubectl port-forward`) plus the credential pair — without editing the
notebook. This is the same env-driven pattern the storage notebooks (`10` / `11`) use.

DuckDB's `s3_endpoint` wants a bare **`host:port`** (no scheme), so we strip the scheme off
`LAKEFS_ENDPOINT`. lakeFS speaks plain HTTP and path-style S3, region `us-east-1` — the
same options `datasets_lake.ipynb` gives `s3fs`.

In [1]:
import os, duckdb, polars as pl, pyarrow as pa

# committed default = in-cluster lakeFS S3 gateway; a run overrides LAKEFS_ENDPOINT via env
LAKEFS_ENDPOINT = os.environ.get("LAKEFS_ENDPOINT", "http://lakefs.data-mesh.svc.cluster.local:8000")
S3_HOST = LAKEFS_ENDPOINT.split("://", 1)[-1]     # duckdb s3_endpoint wants host:port, no scheme

con = duckdb.connect()                             # embedded, in-kernel — no server
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_endpoint='{S3_HOST}'; SET s3_use_ssl=false; SET s3_url_style='path'; SET s3_region='us-east-1';")
con.execute(
    f"SET s3_access_key_id='{os.environ['LAKEFS_ACCESS_KEY_ID']}'; "
    f"SET s3_secret_access_key='{os.environ['LAKEFS_SECRET_ACCESS_KEY']}';"
)

# q(sql) -> run a statement, hand the rows back as a polars DataFrame (house style, as in 10/11/20)
def q(sql):
    return con.execute(sql).pl()

print("lakeFS S3 gateway : connected (endpoint from $LAKEFS_ENDPOINT)")
print("DuckDB version    :", q("SELECT version() AS v")["v"][0])   # proves the engine is live in-kernel

lakeFS S3 gateway : connected (endpoint from $LAKEFS_ENDPOINT)
DuckDB version    : v1.5.5


## Discover the real objects first (never author a path)

Before querying, we **list** what actually lives under `music/main/parquet/` — DuckDB's
`glob()` walks the lakeFS S3 gateway and returns real object keys. We pick our targets from
this listing rather than hard-coding a path that might not exist. The dbt mart
`mart_spotify_audio/part.parquet` is the single Parquet the Dagster `mart_export` writes;
the listing confirms it, alongside the raw dataset mirrors.

In [2]:
objs = q("SELECT file FROM glob('s3://music/main/parquet/*/*.parquet') ORDER BY file")
print(f"{objs.height} Parquet objects on music/main:")
with pl.Config(fmt_str_lengths=80, tbl_rows=20):
    print(objs)

# the analytical target we query below — verified present in the listing above
MART = "s3://music/main/parquet/mart_spotify_audio/part.parquet"
assert objs.filter(pl.col("file") == MART).height == 1, "expected mart object not found in listing"
print("\nquery target:", MART)

29 Parquet objects on music/main:
shape: (29, 1)
┌─────────────────────────────────────────────────────────────────────────┐
│ file                                                                    │
│ ---                                                                     │
│ str                                                                     │
╞═════════════════════════════════════════════════════════════════════════╡
│ s3://music/main/parquet/audioset/test.parquet                           │
│ s3://music/main/parquet/audioset/train.parquet                          │
│ s3://music/main/parquet/fma_echonest/fma_echonest.parquet               │
│ s3://music/main/parquet/fma_features/fma_features.parquet               │
│ s3://music/main/parquet/fma_genres/fma_genres.parquet                   │
│ s3://music/main/parquet/fma_tracks/fma_tracks.parquet                   │
│ s3://music/main/parquet/genre_feast_training/part.parquet               │
│ s3://music/main/parquet/gtzan/gtzan.p

## DuckDB's analytical core — aggregation over live lakeFS Parquet

`read_parquet('s3://…')` streams the object straight from lakeFS into DuckDB's vectorized
engine — no download step, no copy into a local file. Here: the Spotify audio-features mart,
aggregated by genre. DuckDB reads only the columns the query touches.

In [3]:
agg = q(f'''
    SELECT track_genre,
           count(*)                     AS n_tracks,
           round(avg(danceability), 3)  AS danceability,
           round(avg(energy), 3)        AS energy,
           round(avg(valence), 3)       AS valence
    FROM read_parquet('{MART}')
    GROUP BY track_genre
    ORDER BY energy DESC
    LIMIT 10
''')
agg

track_genre,n_tracks,danceability,energy,valence
str,i64,f64,f64,f64
"""death-metal""",901,0.37,0.932,0.25
"""grindcore""",985,0.272,0.926,0.217
"""happy""",995,0.553,0.911,0.328
"""metalcore""",718,0.427,0.902,0.329
"""hardstyle""",898,0.531,0.899,0.31
"""drum-and-bass""",955,0.532,0.876,0.317
"""black-metal""",996,0.297,0.875,0.192
"""heavy-metal""",997,0.428,0.874,0.388
"""party""",820,0.667,0.869,0.676


## Window function — rank genres, two ways at once

DuckDB is a full analytical SQL engine, window functions included. We rank each genre by
its average energy **and** by its average danceability in one pass — a `RANK() OVER (...)`
per metric — so you can read where a genre lands on each axis side by side.

In [4]:
windowed = q(f'''
    WITH g AS (
        SELECT track_genre,
               avg(energy)       AS e,
               avg(danceability) AS d
        FROM read_parquet('{MART}')
        GROUP BY track_genre
    )
    SELECT track_genre,
           round(e, 3)                        AS energy,
           rank() OVER (ORDER BY e DESC)       AS energy_rank,
           round(d, 3)                        AS danceability,
           rank() OVER (ORDER BY d DESC)       AS dance_rank
    FROM g
    ORDER BY energy_rank
    LIMIT 10
''')
windowed

track_genre,energy,energy_rank,danceability,dance_rank
str,f64,i64,f64,i64
"""death-metal""",0.932,1,0.37,106
"""grindcore""",0.926,2,0.272,112
"""happy""",0.911,3,0.553,65
"""metalcore""",0.902,4,0.427,103
"""hardstyle""",0.899,5,0.531,82
"""drum-and-bass""",0.876,6,0.532,81
"""black-metal""",0.875,7,0.297,111
"""heavy-metal""",0.874,8,0.428,102
"""party""",0.869,9,0.667,23


## Projection & predicate pushdown — DuckDB reads only what it needs

DuckDB doesn't slurp the whole Parquet and filter in memory. Its Parquet reader **pushes
the projection and the filter down into the scan**: only the referenced columns are decoded,
and the `WHERE` predicate is evaluated against row-group statistics so non-matching row
groups are skipped entirely.

`EXPLAIN` makes this visible. In the `READ_PARQUET` node below, look for the **`Projections:`**
list (only the columns the query needs) and the **`Filters:`** line (the predicate executing
*inside* the scan). Neither the unused columns nor the filtered-out rows travel up the plan.

In [5]:
plan = con.execute(f'''
    EXPLAIN
    SELECT track_genre, energy
    FROM read_parquet('{MART}')
    WHERE energy > 0.9
''').fetchone()[1]
print(plan)

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│        track_genre        │
│           energy          │
│                           │
│        ~17,948 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        READ_PARQUET       │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│        Projections:       │
│           energy          │
│        track_genre        │
│                           │
│    Filters: energy>0.9    │
│                           │
│        ~17,948 rows       │
└───────────────────────────┘



The `READ_PARQUET` node lists `Projections: energy, track_genre` — the *only* two columns
decoded off disk — and `Filters: energy>0.9` executing in the scan itself. On a wide table
that is the difference between reading two columns and reading them all: DuckDB asks the
Parquet file for exactly what the query needs and nothing more.

## Zero-copy Arrow interop — the embedded engine's superpower

Because DuckDB, polars and pyarrow all speak the **Arrow memory format**, and DuckDB is
running *in this process*, they can hand data back and forth **with no serialization and no
copy** — they point at the *same bytes* in RAM. This is the thing a served engine can never
do: a network hop always serializes.

We show the full round-trip:

1. pull a lakeFS query result out of DuckDB as a **polars** frame (`.pl()`),
2. get its **Arrow** table (`.to_arrow()` — polars→Arrow is zero-copy, same buffers),
3. query that Arrow table **directly in DuckDB by variable name** (DuckDB's *replacement
   scan* reads the Arrow buffers in place — no import, no copy),
4. pull the answer back out as polars / Arrow.

Then we *prove* it's the same memory: the Arrow buffer backing the `energy` column has the
**identical address** whether reached through the polars frame or through its Arrow table.

In [6]:
# 1) DuckDB -> polars: a slice of the mart, straight from lakeFS
frame = con.execute(f'''
    SELECT track_id, track_genre, energy, danceability, valence
    FROM read_parquet('{MART}')
    LIMIT 5000
''').pl()
print("1) polars frame from lakeFS :", frame.shape)

# 2) polars -> Arrow (zero-copy: the Arrow table shares polars' buffers)
audio_arrow = frame.to_arrow()
print("2) Arrow table (same bytes) :", audio_arrow.num_rows, "rows,", audio_arrow.num_columns, "cols")

# 3) DuckDB queries the Arrow table BY NAME -- replacement scan reads it in place, no copy
regrouped = con.execute('''
    SELECT track_genre, count(*) AS n, round(avg(energy), 3) AS avg_energy
    FROM audio_arrow                     -- <- an in-process Arrow table, scanned zero-copy
    GROUP BY track_genre
    ORDER BY n DESC
    LIMIT 5
''').pl()                                 # 4) ...and back out to polars
print("3-4) DuckDB over the Arrow table, back to polars:")
print(regrouped)

# and back out as an Arrow table too (DuckDB's other zero-copy bridge)
summary_tbl = con.execute("SELECT round(avg(valence), 4) AS avg_valence, count(*) AS n FROM audio_arrow").to_arrow_table()
print("\nresult pulled back as Arrow:", summary_tbl.to_pydict())

# PROOF of zero-copy: same buffer address via polars and via its Arrow table
addr_via_arrow  = audio_arrow.column("energy").chunk(0).buffers()[-1].address
addr_via_polars = frame["energy"].to_arrow().buffers()[-1].address
print("\nenergy-column buffer address (via Arrow) :", addr_via_arrow)
print("energy-column buffer address (via polars):", addr_via_polars)
print("same physical memory -- zero copy        :", addr_via_arrow == addr_via_polars)

1) polars frame from lakeFS : (5000, 5)
2) Arrow table (same bytes) : 5000 rows, 5 cols


3-4) DuckDB over the Arrow table, back to polars:
shape: (5, 3)
┌─────────────┬─────┬────────────┐
│ track_genre ┆ n   ┆ avg_energy │
│ ---         ┆ --- ┆ ---        │
│ str         ┆ i64 ┆ f64        │
╞═════════════╪═════╪════════════╡
│ acoustic    ┆ 84  ┆ 0.428      │
│ malay       ┆ 76  ┆ 0.615      │
│ alt-rock    ┆ 75  ┆ 0.71       │
│ grindcore   ┆ 74  ┆ 0.928      │
│ new-age     ┆ 73  ┆ 0.207      │
└─────────────┴─────┴────────────┘

result pulled back as Arrow: {'avg_valence': [0.4798], 'n': [5000]}

energy-column buffer address (via Arrow) : 101887891187808
energy-column buffer address (via polars): 101887891187808
same physical memory -- zero copy        : True


**Why it matters:** in the embedded model, moving a 5,000-row (or 5,000,000-row) frame
between polars, pyarrow and DuckDB costs *nothing* — no encode, no decode, no bytes moved.
They are three lenses over one Arrow buffer. That is what "embedded, in-process" buys you,
and it's exactly what disappears the moment the engine lives across a socket — which is the
trade the served half makes on purpose, next.

# Half 2 — GizmoSQL: served DuckDB over Arrow Flight SQL

Same engine, opposite shape. **GizmoSQL** wraps DuckDB in an **Arrow Flight SQL** server at a
real `host:port`, so instead of one process owning the database, *many* clients share one
DuckDB over the network — BI tools, IDEs (DataGrip via the Arrow Flight SQL JDBC driver),
and pipelines all hit the same endpoint. It serves the mesh's silver as **persisted DuckDB
base tables** (one per lakeFS Parquet file) in the `datasets_music` and `datasets_health`
schemas, so queries hit native columnar storage instead of re-reading Parquet each time.

The trade vs Half 1 is exactly the zero-copy we just proved: a Flight SQL result crosses a
socket, so it's serialized to Arrow IPC on the wire (still Arrow, still efficient — but a
copy). What you get back is a *shared, always-on* engine.

## Setup — the ADBC Flight SQL client (not baked)

The embedded half needed no install. The **served** half does: the ADBC Flight SQL driver is
**not** in the singleuser image, so we install it here. (`polars` — for rendering — already
ships in the image.)

In [7]:
%pip install -q adbc-driver-flightsql adbc-driver-manager


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect — Arrow Flight SQL over plaintext gRPC

Connection is **env-driven**, mirroring the Dagster catalog emitter exactly. The committed
default is the **in-cluster** GizmoSQL service (`gizmosql.data-mesh.svc.cluster.local:31337`);
a validation run overrides `GIZMOSQL_URI` and the credentials via env.

GizmoSQL runs **TLS-off** (`grpc+tcp://`, plaintext) — in the cluster the hop is secured by
**Istio mTLS**, and the LAN NodePort is trusted (this is a $0 LAN-only lab). We render results
via polars to match house style.

In [8]:
import os
import adbc_driver_flightsql.dbapi as flight_sql

GIZMOSQL_URI = os.environ.get("GIZMOSQL_URI", "grpc+tcp://gizmosql.data-mesh.svc.cluster.local:31337")

conn = flight_sql.connect(
    GIZMOSQL_URI,
    db_kwargs={
        "username": os.environ.get("GIZMOSQL_USERNAME", "weyland"),
        "password": os.environ["GIZMOSQL_PASSWORD"],   # from the gizmosql-secret k8s Secret
    },
)

# g(sql) -> run a statement over Flight SQL, hand the rows back as a polars DataFrame
def g(sql):
    cur = conn.cursor()
    try:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
        return pl.DataFrame(rows, schema=cols, orient="row")
    finally:
        cur.close()

print("GizmoSQL          : connected via Arrow Flight SQL (endpoint from $GIZMOSQL_URI)")
print("served DuckDB     :", g("SELECT version() AS v")["v"][0])   # DuckDB, reached over the wire

GizmoSQL          : connected via Arrow Flight SQL (endpoint from $GIZMOSQL_URI)
served DuckDB     : v1.5.4


## Explore — what tables are served?

GizmoSQL surfaces the silver as base tables; `information_schema.tables` lists them. We look
at the two mesh schemas — `datasets_music` and `datasets_health` — and count what's there.
(The Flight SQL `GetTables` metadata an IDE's tree is built from surfaces exactly these base
tables.)

In [9]:
served = g('''
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema LIKE 'datasets_%'
    ORDER BY table_schema, table_name
''')
counts = served.group_by("table_schema").len().sort("table_schema")
print("base tables served per schema:")
print(counts)
print(f"\n{served.height} tables total. A sample from each schema:")
for schema in counts["table_schema"]:
    sample = served.filter(pl.col("table_schema") == schema)["table_name"].to_list()[:6]
    print(f"  {schema}: {', '.join(sample)} …")

base tables served per schema:
shape: (2, 2)
┌─────────────────┬─────┐
│ table_schema    ┆ len │
│ ---             ┆ --- │
│ str             ┆ u32 │
╞═════════════════╪═════╡
│ datasets_health ┆ 67  │
│ datasets_music  ┆ 45  │
└─────────────────┴─────┘

112 tables total. A sample from each schema:
  datasets_health: big_five_big5_data, brfss_brfss_2020, brfss_brfss_prevalence_2011_present, brfss_brfss_prevalence_data, brfss_brfss_selected_metro, brfss_brfss_selected_metropolitan_area …
  datasets_music: audioset_test, audioset_train, fma_echonest, fma_echonest_1782666876_6288455_44631944d3, fma_echonest_1782666876_6288455_646de4fd71, fma_echonest_1782678943_68445_f142f37f70 …


## A real relational JOIN — DuckDB does true joins

Unlike the wide-column / key-value Tier-2 stores, DuckDB is a real relational engine — so
GizmoSQL serves real JOINs. The USDA FoodData Central set is relational: a `food` table and a
`food_nutrient` table that meet on `fdc_id`. We join them to rank foods by how many nutrient
measurements they carry.

The materialised base-table names are long and underscored (one per source CSV), so we
**resolve the exact names from `information_schema` first** and only then build the query —
never trusting a literal name that the catalog might disagree with.

In [10]:
import os.path
health = served.filter(pl.col("table_schema") == "datasets_health")["table_name"].to_list()

# resolve the two USDA tables from what is ACTUALLY served. The base-table names are the source
# CSV names, so they share a common prefix (…_2024_10_31_); the two we want end exactly in
# `food` and `food_nutrient` (not `branded_food`, `foundation_food`, …), so match on prefix+leaf.
usda = [t for t in health if t.startswith("usda_fooddata")]
usda_prefix  = os.path.commonprefix(usda)
food_tbl     = next(t for t in usda if t == usda_prefix + "food")
nutrient_tbl = next(t for t in usda if t == usda_prefix + "food_nutrient")
print("resolved food table     :", food_tbl)
print("resolved nutrient table :", nutrient_tbl)

most_measured = g(f'''
    SELECT f.description, count(*) AS nutrients
    FROM datasets_health.{food_tbl} f
    JOIN datasets_health.{nutrient_tbl} fn USING (fdc_id)
    GROUP BY f.description
    ORDER BY nutrients DESC
    LIMIT 20
''')
print(f"\ntop 20 most-measured foods (JOIN across two base tables on fdc_id):")
most_measured

resolved food table     : usda_fooddata_fooddata_central_csv_2024_10_31_food
resolved nutrient table : usda_fooddata_fooddata_central_csv_2024_10_31_food_nutrient



top 20 most-measured foods (JOIN across two base tables on fdc_id):


description,nutrients
str,i64
"""ICE CREAM""",35132
"""Plastic Bottle""",31500
"""2% REDUCED FAT MILK""",25218
"""POTATO CHIPS""",23794
"""CUT GREEN BEANS""",19502
…,…
"""SOUR CREAM""",11528
"""Aluminum Can""",11372
"""TOMATO SAUCE, TOMATO""",11364


## A couple more, from what's actually present

Two small aggregations over other served tables — a WHO Global Health Observatory table in
`datasets_health`, and the Spotify audio mart in `datasets_music` (the *same* data Half 1 read
as Parquet, here served as a persisted base table). We resolve each table name from the
served list before querying it.

In [11]:
# WHO GHO -- latest adult-obesity value per country (max_by picks the value at the latest year)
who_tbl = next(t for t in health if t == "who_gho_adult_obesity")
who = g(f'''
    SELECT spatialdim AS country,
           round(max_by(numericvalue, timedim), 1) AS latest_obesity_pct
    FROM datasets_health.{who_tbl}
    GROUP BY spatialdim
    ORDER BY latest_obesity_pct DESC
    LIMIT 10
''')
print("WHO GHO -- highest adult-obesity prevalence (latest year per country):")
print(who)

# datasets_music -- the Spotify audio mart, served as a base table (same data as Half 1's Parquet)
music = served.filter(pl.col("table_schema") == "datasets_music")["table_name"].to_list()
mart_tbl = next(t for t in music if t.startswith("mart_spotify_audio"))
energy = g(f'''
    SELECT track_genre,
           count(*)               AS n_tracks,
           round(avg(energy), 3)  AS energy
    FROM datasets_music.{mart_tbl}
    GROUP BY track_genre
    ORDER BY energy DESC
    LIMIT 10
''')
print(f"\ndatasets_music.{mart_tbl} -- highest-energy genres (served base table):")
print(energy)

WHO GHO -- highest adult-obesity prevalence (latest year per country):
shape: (10, 2)
┌─────────┬────────────────────┐
│ country ┆ latest_obesity_pct │
│ ---     ┆ ---                │
│ str     ┆ f64                │
╞═════════╪════════════════════╡
│ NRU     ┆ 71.4               │
│ ASM     ┆ 70.6               │
│ COK     ┆ 70.6               │
│ TKL     ┆ 66.7               │
│ TON     ┆ 63.9               │
│ NIU     ┆ 63.2               │
│ TUV     ┆ 57.7               │
│ WSM     ┆ 51.7               │
│ PYF     ┆ 47.4               │
│ USA     ┆ 39.6               │
└─────────┴────────────────────┘

datasets_music.mart_spotify_audio_part -- highest-energy genres (served base table):
shape: (10, 3)
┌───────────────┬──────────┬────────┐
│ track_genre   ┆ n_tracks ┆ energy │
│ ---           ┆ ---      ┆ ---    │
│ str           ┆ i64      ┆ f64    │
╞═══════════════╪══════════╪════════╡
│ death-metal   ┆ 901      ┆ 0.932  │
│ grindcore     ┆ 985      ┆ 0.926  │
│ happy         ┆ 9

> **Read-only, isolated sessions.** GizmoSQL runs each client statement in an **isolated
> session**, so even a `CREATE`/`INSERT` from a client wouldn't persist into the served
> database — but we never attempt one. Everything above is `SELECT` over base tables the
> mesh materialised. Nothing to clean up.

## When to reach for each — embedded vs served vs Trino

Both halves are the *same DuckDB engine*; the difference is **where it runs and who shares it**.

| | **Embedded DuckDB** (Half 1) | **Served — GizmoSQL** (Half 2) |
|---|---|---|
| Runs | **in your process** (this kernel) | as a **server** at a `host:port` |
| Clients | one — the process it lives in | **many** — BI tools, IDEs, pipelines share it |
| Data it reads here | lakeFS **Parquet**, live, over `httpfs` | **persisted DuckDB base tables** (silver, one per Parquet) |
| Arrow interop | **zero-copy, same memory** (polars ↔ Arrow ↔ DuckDB) | Arrow over the wire (Flight SQL IPC — a copy) |
| Sessions | your session, stateful in-kernel | **isolated per statement** on the server |
| Access path | Python `duckdb` in-process | **Arrow Flight SQL** (ADBC in-pod; JDBC for DataGrip) |
| Great for | notebooks, edge, ad-hoc, single-process ETL | shared querying, BI dashboards, IDE browsing |
| Not for | sharing across clients / tools | the zero-copy in-process hot path |

**Reach for embedded DuckDB** when the work is *in your process* — a notebook, an edge job, a
one-shot transform — and you want the columnar engine and zero-copy Arrow with **no service to
run**. **Reach for GizmoSQL** when *several* clients need the same DuckDB — a BI tool, DataGrip
over the Arrow Flight SQL JDBC driver, a pipeline — served from one always-on endpoint over
persisted tables.

And to place this whole notebook against **`20`**: **Trino is the *distributed-federation* half**
of the query layer — one query spanning many heterogeneous engines, cross-catalog joins, no ETL.
**DuckDB / GizmoSQL is the *single-node OLAP* half** — one fast columnar engine, embedded in your
process or served to a few clients. Federate many stores → Trino. Drive one columnar engine hard,
in-process or shared → DuckDB, the two ways above.